# Supervisor / Orchestrator Pattern: Practical Demo

One supervisor, two specialist workers, and **four supervisor variants**, to see decomposition quality and static-vs-dynamic delegation as real, visible differences — not just description.

**The supervisor's four jobs** (from our notes): it receives the task, **decomposes** it into sub-tasks, **delegates** each sub-task to the right specialist worker, **collects** each worker's output, and **synthesises** a final answer from all of them. Workers never talk to each other — everything goes through the supervisor.

**Two specialist workers:**
- **Billing worker:** answers from a real, short billing-policy text.
- **Research worker:** uses a real, live web-search tool for current facts.

**Four supervisor variants**, same LLM, only the supervisor's own instructions differ:

| Variant | Decomposition | Delegation timing |
|---|---|---|
| Static + good | writes a clear, scoped sub-task per worker | plans the full worker sequence upfront, in one call |
| Static + bad | forwards the user's full, unedited question to every worker | plans the full worker sequence upfront, in one call |
| Dynamic + good | writes a clear, scoped sub-task per worker | decides one worker at a time, after seeing each result |
| Dynamic + bad | forwards the user's full, unedited question to every worker | decides one worker at a time, after seeing each result |

**What we will do**
1. Setup
2. The two specialist workers
3. The static supervisor (good and bad)
4. The dynamic supervisor (good and bad), as a LangGraph loop
5. Run two real queries through all four variants
6. Comparison and summary

## 1. Setup

In [ ]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [ ]:
pip install langchain-groq langgraph langchain-community ddgs -q # q for silent installation

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

## 2. The two specialist workers

Each worker is just a function: it gets a **sub-task** (written by the supervisor) and returns its own answer, using only its own tool or knowledge.

In [ ]:
# Real billing policy (the billing worker's only source of truth)
billing_policy = """ShopSmart Billing Policy:
- Monthly plans: cancel anytime, no refund for the current month.
- Yearly plans: cancel anytime, refund the unused months, minus a 10% processing fee.
- Refunds are processed within 5-7 business days to the original payment method."""

def billing_worker(sub_task):
    prompt = f"""You are the billing specialist. Answer ONLY using the policy below.

Policy:
{billing_policy}

Sub-task: {sub_task}"""
    return llm.invoke(prompt).content

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()  # a real, live web search tool

from ddgs.exceptions import DDGSException

def research_worker(sub_task):
    try:
        results = search.run(sub_task)
    except DDGSException as e:
        results = f"(Web search unavailable right now — {e}. No search results to go on.)"

    prompt = f"""You are the research specialist. Answer the sub-task using the search results below.
If the search results say no information is available, say that clearly instead of guessing.

Search results:
{results}

Sub-task: {sub_task}"""
    return llm.invoke(prompt).content

In [ ]:
workers = {'billing': billing_worker, 'research': research_worker}

## 3. Static supervisor

The supervisor decides the **entire** worker sequence and every sub-task **upfront**, in a single LLM call. It does not look at any worker's output before planning. Once the plan is made, it just executes it, collects every output, and synthesises the final answer.

### 3.1 Decomposition: good vs bad

The only difference between "good" and "bad" is this one instruction.

In [ ]:
good_instruction = "For each worker, write a short, clear sub-task containing only the part of the user's request that worker needs."
bad_instruction = "For each worker, just copy the user's full question unchanged as the sub-task, with no rewriting."

In [ ]:
def static_plan(user_query, style):
    instruction = good_instruction if style == 'good' else bad_instruction

    prompt = f"""You are a supervisor managing two workers: "billing" (billing policy questions)
and "research" (live web search for current facts).

{instruction}

Decide which worker(s) are needed for the user's request, in order.
Write ONE line per worker, in exactly this format, and nothing else:
worker_name | sub-task text

User request: {user_query}"""

    lines = llm.invoke(prompt).content.strip().splitlines()
    plan = []
    for line in lines:
        if '|' in line:
            worker, sub_task = line.split('|', 1)
            worker = worker.strip().lower()
            if worker in workers:
                plan.append((worker, sub_task.strip()))
    return plan

### 3.2 Delegate, collect, synthesise

In [ ]:
def static_supervisor(user_query, style):
    plan = static_plan(user_query, style)                         # decompose (1 LLM call)
    calls = 1

    print(f"PLAN ({style}):")
    for worker, sub_task in plan:
        print(f"  -> {worker}: {sub_task}")

    outputs = {}
    for worker, sub_task in plan:                                  # delegate + collect
        outputs[worker] = workers[worker](sub_task)
        print(f"  [{worker} worker] {outputs[worker][:150]}...")

    # synthesise (1 LLM call)
    combined = '\n'.join(f'{w}: {o}' for w, o in outputs.items())
    synthesis_prompt = f"""Combine the specialist answers below into one clear reply to the user's request.

User request: {user_query}

Specialist answers:
{combined}"""
    final_answer = llm.invoke(synthesis_prompt).content
    calls += 1

    return final_answer, calls

## 4. Dynamic supervisor (LangGraph)

The supervisor decides **one worker at a time**, after seeing every previous result, and stops when it decides it has enough. This is a loop, so we build it as a LangGraph graph, the same way as our other agent notebooks.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    user_query: str
    style: str               # 'good' or 'bad'
    collected: Annotated[list, lambda a, b: a + b]  # (worker, sub_task, output) so far
    calls: int
    next_step: str            # the supervisor's latest decision, e.g. "billing | ..." or "DONE"
    final_answer: str         # the synthesised answer, set only at the end

In [ ]:
def supervisor_node(state):
    instruction = good_instruction if state['style'] == 'good' else bad_instruction
    done_so_far = '\n'.join(f"{w}: {o}" for w, _, o in state['collected']) or '(nothing yet)'

    prompt = f"""You are a supervisor managing two workers: "billing" (billing policy questions)
and "research" (live web search for current facts).

{instruction}

User request: {state['user_query']}

Results collected so far:
{done_so_far}

Decide the SINGLE next worker to call, and its sub-task. If you already have everything
needed to answer the user fully, respond with exactly: DONE
Otherwise respond in exactly this format, and nothing else:
worker_name | sub-task text"""

    output = llm.invoke(prompt).content.strip()
    print(f"  SUPERVISOR decided: {output}")
    return {'calls': state['calls'] + 1, 'next_step': output}

In [ ]:
def worker_node(state):
    worker, sub_task = state['next_step'].split('|', 1)
    worker, sub_task = worker.strip().lower(), sub_task.strip()

    output = workers[worker](sub_task)
    print(f"  [{worker} worker] {output[:150]}...")
    return {'collected': [(worker, sub_task, output)]}

In [ ]:
def route(state) -> Literal['worker', 'synthesise']:
    if state['next_step'].strip().upper() == 'DONE' or len(state['collected']) >= 3:  # simple step guard
        return 'synthesise'
    return 'worker'

In [ ]:
def synthesise_node(state):
    combined = '\n'.join(f"{w}: {o}" for w, _, o in state['collected'])
    prompt = f"""Combine the specialist answers below into one clear reply to the user's request.

User request: {state['user_query']}

Specialist answers:
{combined}"""
    answer = llm.invoke(prompt).content
    return {'calls': state['calls'] + 1, 'final_answer': answer}

In [ ]:
builder = StateGraph(State)
builder.add_node('supervisor', supervisor_node)
builder.add_node('worker', worker_node)
builder.add_node('synthesise', synthesise_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route)
builder.add_edge('worker', 'supervisor')
builder.add_edge('synthesise', END)

dynamic_graph = builder.compile()

In [ ]:
def dynamic_supervisor(user_query, style):
    print(f"DYNAMIC RUN ({style}):")
    initial_state = {'user_query': user_query, 'style': style, 'collected': [], 'calls': 0}
    final_state = dynamic_graph.invoke(initial_state)
    return final_state['final_answer'], final_state['calls']

## 5. Run two real queries through all four variants

- **Q1 (simple):** needs only one worker. Decomposition quality should barely matter.
- **Q2 (compound):** needs both workers, in a way that should expose bad (unscoped) delegation — each worker should only need its own part of the question, not the whole thing.

In [ ]:
queries = {
    'Q1 (simple)': 'What is your refund policy for a yearly plan?',
    'Q2 (compound)': ('If I cancel my yearly subscription today, what refund do I get — '
                       'and is there any current OpenAI API outage I should know about first?'),
}

In [ ]:
results = {}

for label, query in queries.items():
    print('=' * 70)
    print(label, ':', query)
    print('=' * 70)

    for mode, run_fn in [('static', static_supervisor), ('dynamic', dynamic_supervisor)]:
        for style in ['good', 'bad']:
            print(f"\n--- {mode} + {style} ---")
            answer, calls = run_fn(query, style)
            results[(label, mode, style)] = {'answer': answer, 'calls': calls}
            print(f"FINAL ANSWER: {answer}")
            print(f"Supervisor LLM calls used: {calls}")

Q1 (simple) : What is your refund policy for a yearly plan?

--- static + good ---
PLAN (good):
  -> billing: What is your refund policy for a yearly plan?
  [billing worker] For a yearly plan, you can cancel at any time. We’ll refund the unused months, minus a 10% processing fee. The refund will be processed within 5‑7 bus...
FINAL ANSWER: **Refund policy for a yearly plan**

- You can cancel your yearly subscription at any time.  
- We’ll refund the portion of the plan that you haven’t used, minus a 10 % processing fee.  
- The refund will be credited back to the original payment method within 5–7 business days.
Supervisor LLM calls used: 2

--- static + bad ---
PLAN (bad):
  -> billing: What is your refund policy for a yearly plan?
  [billing worker] For a yearly plan, you can cancel at any time. We will refund the unused months, minus a 10% processing fee. The refund will be processed within 5‑7 b...
FINAL ANSWER: **Refund policy for a yearly plan**

- You may cancel your yearly su

## 6. Comparison and summary

In [ ]:
import pandas as pd

rows = []
for (label, mode, style), r in results.items():
    rows.append({'query': label, 'mode': mode, 'style': style, 'calls': r['calls'], 'answer': r['answer'][:200]})

pd.DataFrame(rows)

,query,mode,style,calls,answer
0,Q1 (simple),static,good,2,**Refund policy for a yearly plan**\n\n- You c...
1,Q1 (simple),static,bad,2,**Refund policy for a yearly plan**\n\n- You m...
2,Q1 (simple),dynamic,good,3,"If you cancel a yearly plan, you’ll receive a ..."
3,Q1 (simple),dynamic,bad,3,**Refund policy for a yearly plan**\n\n- You m...
4,Q2 (compound),static,good,2,"If you cancel your yearly plan today, you’ll r..."
5,Q2 (compound),static,bad,2,**What happens if you cancel your yearly ChatG...
6,Q2 (compound),dynamic,good,4,**Refund for canceling a yearly plan**\n\n- Yo...
7,Q2 (compound),dynamic,bad,4,**Refund for canceling a yearly subscription**...


**How to read this table:**
- For **Q1**, the `good` and `bad` answers should be similar — decomposition barely matters for a one-worker task.
- For **Q2**, compare `good` vs `bad` within the same mode: did the bad (unscoped) version confuse a worker, by sending it a sub-task it didn't need? Read the printed sub-task lines above the table to check directly.
- Compare the `calls` column between `static` and `dynamic`: static should use exactly 2 calls (plan + synthesise) regardless of how many workers it used; dynamic uses one call per worker plus one synthesis call, so it costs more for the compound query.

### Summary, from our notes

**Why multi-agent at all:** context limits (each worker gets a clean, focused context instead of one agent doing everything), specialisation (a billing-only prompt and a research-only prompt each do their one job better than one agent doing both), and parallelism (not demonstrated here, since both our runs call workers one at a time).

**The supervisor's hardest problem is decomposition.** This notebook makes that concrete: the "bad" supervisor's unscoped sub-tasks are the same failure mode as writing a vague or oversized prompt for a worker, which our notes say degrades the worker's output and the final synthesis.

**Static vs dynamic:** static is cheaper and faster (one planning call, no re-checking) but brittle if an early assumption about what's needed turns out wrong — it can't react to what a worker actually returns. Dynamic costs one extra LLM call per worker, but can re-evaluate after every result, which is why our notes call it more flexible but slower.